# Spectral-MAE: final study

Compares **Vanilla MAE** with **Spectral-MAE** (brightness-weighted NDVI loss, λ = 10, summed MSE as in the spec),
3 pre-training seeds each, plus the random-encoder probe and the supervised-from-scratch baseline.

**Before running:**
1. *Add Input → Upload*: upload the **new** `spectral-mae.zip` (it has the brightness-weighting option). Remove the old input.
2. *Session options*: **GPU T4 or P100**, **Internet On**.
3. **Save Version → Save & Run All (Commit)** so it runs in the background (about 4 h at 100 epochs).

In [1]:
# 1. Copy the code (picks the upload that has the brightness-weighting option)
import glob, os, shutil, subprocess
cands = glob.glob('/kaggle/input/**/configs/default.yaml', recursive=True)
if not cands:
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        subprocess.run(['unzip', '-q', '-o', z, '-d', '/kaggle/tmp_code'], check=True)
    cands = glob.glob('/kaggle/tmp_code/**/configs/default.yaml', recursive=True)
new = [c for c in cands if 'ndvi_weighting' in open(c).read()]
assert new, 'Upload the NEW spectral-mae.zip (the old one has no ndvi_weighting option)'
repo = os.path.dirname(os.path.dirname(new[0]))
shutil.copytree(repo, '/kaggle/working/spectral-mae', dirs_exist_ok=True)
os.chdir('/kaggle/working/spectral-mae')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['EPOCHS'] = '100'   # pre-training epochs; 200 is better but doubles the time
print('Code at', os.getcwd())

Code at /kaggle/working/spectral-mae


In [2]:
# 2. Dependencies, GPU check, unit tests
!pip install -q datasets tifffile
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python -m pytest -q

CUDA: True Tesla T4
...............F                                                         [100%]
=================================== FAILURES ===================================
______________ test_brightness_weighting_downweights_dark_pixels _______________

    def test_brightness_weighting_downweights_dark_pixels():
        true = torch.full((1, 4, 1, 2), 0.2)
        true[0, :, 0, 0] = 0.02                 # pixel 0 is dark (NIR+Red = 0.04), pixel 1 bright (0.4)
        pred = true.clone()
        pred[0, 3, 0, 0] *= 0.5                 # NDVI error only on the dark pixel
        mask = torch.ones(1, 1, 1, 2)
        plain = NDVIConsistencyLoss(2, 3)(pred, true, mask).item()
        bright = NDVIConsistencyLoss(2, 3, weighting="brightness")(pred, true, mask).item()
        assert plain == pytest.approx(1 / 6, abs=1e-3)
>       assert bright == pytest.approx(0.04 * (1 / 3) / 0.44, abs=1e-3)
E       assert 0.01333288848400116 == 0.0303030303030303 ± 0.001
E         
E         comp

In [3]:
# 3. Data (check: 'Band order looks correct.' and 'train 18898, val 2700, test 5400')
!python -m spectral_mae.prep_data --from-hf

README.md: 100%|███████████████████████████| 4.56k/4.56k [00:00<00:00, 1.44MB/s]

data/train-00000-of-00005.parquet: downloading bytes: █████▌|  291MB, 25.2MB/s  
data/train-00000-of-00005.parquet: downloading bytes: ██████|  305MB, 28.0MB/s  
data/train-00000-of-00005.parquet: reconstructing file: 100%|█|  314MB /  314MB,

data/train-00001-of-00005.parquet: downloading bytes: ████▊ |  257MB, 22.6MB/s  
data/train-00001-of-00005.parquet: downloading bytes: █████▌|  297MB, 24.7MB/s  
data/train-00001-of-00005.parquet: downloading bytes: ██████|  311MB, 27.0MB/s  
data/train-00001-of-00005.parquet: reconstructing file: 100%|█|  319MB /  319MB,

data/train-00002-of-00005.parquet: downloading bytes: █████▋|  297MB, 26.1MB/s  
data/train-00002-of-00005.parquet: downloading bytes: ██████|  303MB, 28.5MB/s  
data/train-00002-of-00005.parquet: reconstructing file: 100%|█|  312MB /  312MB,

data/train-00003-of-00005.parquet: downloading bytes: ████▊ |  255MB, 16.5MB/s  
data/train-00003-of-0000

In [4]:
%%bash
# 4. Pre-training: Vanilla MAE and Spectral-MAE, seeds 0-2
COMMON="--set loss.mse_reduction=sum_per_patch --set pretrain.epochs=$EPOCHS --set pretrain.warmup_epochs=10"
for s in 3 4 5; do
  python -m spectral_mae.pretrain --name final_vanilla_s$s --set loss.lambda_ndvi=0 --set seed=$s $COMMON
  python -m spectral_mae.pretrain --name final_spectral_s$s --set loss.lambda_ndvi=10 --set loss.ndvi_weighting=brightness --set seed=$s $COMMON
done

Device cuda | params {'total': 3208384, 'encoder': 2866560, 'decoder': 341824} | encoder MACs/img masked 13,328,640 vs full 45,787,392
epoch   1/100 | loss 160.1857 mse 160.1857 ndvi 0.7069 | val mse 0.1130 ndvi_mae 0.7184 | 24.0s
epoch   2/100 | loss 79.7365 mse 79.7365 ndvi 0.6916 | val mse 0.0464 ndvi_mae 0.6314 | 9.4s
epoch   3/100 | loss 25.5276 mse 25.5276 ndvi 0.4903 | val mse 0.0101 ndvi_mae 0.3638 | 9.4s
epoch   4/100 | loss 5.7198 mse 5.7198 ndvi 0.2905 | val mse 0.0039 ndvi_mae 0.2599 | 9.3s
epoch   5/100 | loss 3.4263 mse 3.4263 ndvi 0.2486 | val mse 0.0029 ndvi_mae 0.2378 | 9.5s
epoch   6/100 | loss 2.4887 mse 2.4887 ndvi 0.2147 | val mse 0.0022 ndvi_mae 0.1890 | 9.4s
epoch   7/100 | loss 2.0203 mse 2.0203 ndvi 0.1715 | val mse 0.0019 ndvi_mae 0.1539 | 8.9s
epoch   8/100 | loss 1.8171 mse 1.8171 ndvi 0.1502 | val mse 0.0018 ndvi_mae 0.1409 | 9.1s
epoch   9/100 | loss 1.7549 mse 1.7549 ndvi 0.1434 | val mse 0.0018 ndvi_mae 0.1565 | 9.2s
epoch  10/100 | loss 1.7472 mse 1.747

In [5]:
%%bash
# 5. Linear probes (3 probe seeds each): [CLS] token (spec) AND mean-pooled patch tokens,
#    both declared before seeing results; plus random-encoder references for both.
for ck in runs/final_*/last.pt; do
  python -m spectral_mae.probe --ckpt "$ck"
  python -m spectral_mae.probe --ckpt "$ck" --set probe.pool=mean
done

Extracting cls features on cuda ...
  labels  10.0% ( 1890) seed 0: acc 67.48  macro-F1 65.88
  labels  50.0% ( 9449) seed 0: acc 71.13  macro-F1 70.13
  labels 100.0% (18898) seed 0: acc 72.41  macro-F1 71.64
  labels  10.0% ( 1890) seed 1: acc 67.31  macro-F1 66.20
  labels  50.0% ( 9449) seed 1: acc 70.91  macro-F1 70.11
  labels 100.0% (18898) seed 1: acc 72.43  macro-F1 71.68
  labels  10.0% ( 1890) seed 2: acc 67.19  macro-F1 65.98
  labels  50.0% ( 9449) seed 2: acc 70.78  macro-F1 69.85
  labels 100.0% (18898) seed 2: acc 72.43  macro-F1 71.67

### Linear probe: probe_final_spectral_s3_20261116-072653

| Labels | Seeds | top1_acc | macro_f1 |
|---|---|---|---|
| 10% | 3 | 67.33 ± 0.15 | 66.02 ± 0.16 |
| 50% | 3 | 70.94 ± 0.18 | 70.03 ± 0.15 |
| 100% | 3 | 72.42 ± 0.01 | 71.66 ± 0.02 |

Saved runs/probe_final_spectral_s3_20261116-072653_20261116-084534/results.md
Extracting mean features on cuda ...
  labels  10.0% ( 1890) seed 0: acc 67.44  macro-F1 66.15
  labels  50.0% ( 9449

In [6]:
%%bash
# 6. Supervised ViT from scratch (10/50/100% labels x 3 seeds)
#python -m spectral_mae.supervised

In [7]:
%%bash
# 7. Per-class gradient diagnostics (seed 0)
for ck in runs/final_*_s3_*/last.pt; do python -m spectral_mae.diagnose_gradients --ckpt "$ck"; done

| Class | n | mean NIR+Red | NDVI err | grad MSE | grad NDVI | NDVI/MSE |
|---|---|---|---|---|---|---|
| AnnualCrop | 300 | 0.450 | 0.126 | 0.000723 | 0.0282 | 39.07 |
| Forest | 300 | 0.305 | 0.033 | 0.000389 | 0.0318 | 81.85 |
| HerbaceousVegetation | 300 | 0.341 | 0.058 | 0.000442 | 0.0279 | 63.09 |
| Highway | 250 | 0.342 | 0.146 | 0.000766 | 0.0296 | 38.63 |
| Industrial | 250 | 0.347 | 0.157 | 0.00107 | 0.0278 | 25.99 |
| Pasture | 200 | 0.409 | 0.078 | 0.000636 | 0.0321 | 50.46 |
| PermanentCrop | 250 | 0.410 | 0.091 | 0.000566 | 0.0276 | 48.86 |
| Residential | 300 | 0.330 | 0.123 | 0.000607 | 0.0274 | 45.23 |
| River | 250 | 0.273 | 0.266 | 0.00107 | 0.0317 | 29.63 |
| SeaLake | 300 | 0.085 | 0.063 | 0.00018 | 0.0266 | 147.94 |

Saved runs/diag_final_spectral_s3_20261116-072653_20261116-085414/per_class_gradients.png
| Class | n | mean NIR+Red | NDVI err | grad MSE | grad NDVI | NDVI/MSE |
|---|---|---|---|---|---|---|
| AnnualCrop | 300 | 0.450 | 0.132 | 0.000703 | 0.0637 | 

In [8]:
# 8. Final tables
import json, glob, re
import numpy as np
from collections import defaultdict

def ms(v, d=2):
    v = np.asarray(v, dtype=float)
    return f"{v.mean():.{d}f} ± {v.std(ddof=1) if len(v) > 1 else 0:.{d}f}"

def probe_by_seed(key, prefix='probe'):
    acc, f1 = defaultdict(dict), defaultdict(dict)
    for f in glob.glob(f'runs/{prefix}_{key}_s*/results.json'):
        s = int(re.search(rf'{prefix}_{key}_s(\d+)_', f).group(1))
        res = json.load(open(f))
        for frac in (0.1, 0.5, 1.0):
            acc[frac][s] = 100 * np.mean([r['top1_acc'] for r in res if r['frac'] == frac])
            f1[frac][s] = 100 * np.mean([r['macro_f1'] for r in res if r['frac'] == frac])
    return acc, f1

for prefix, title in [('probe', '[CLS] token probe (spec)'), ('meanprobe', 'Mean-pooled token probe')]:
    print(f'\n### {title}\n')
    print("| Model | Val MSE (×1e-3) | Val NDVI err | Acc 10% | Acc 50% | Acc 100% | Macro-F1 10% |")
    print("|---|---|---|---|---|---|---|")
    rand = glob.glob(f'runs/{prefix}_random_init_*/results.json')
    if rand:
        res = json.load(open(rand[0]))
        cells = [ms([100 * r['top1_acc'] for r in res if r['frac'] == f]) for f in (0.1, 0.5, 1.0)]
        f1 = ms([100 * r['macro_f1'] for r in res if r['frac'] == 0.1])
        print(f"| Random encoder (probe) | – | – | {' | '.join(cells)} | {f1} |")
    sup = glob.glob('runs/supervised_*/results.json')
    if sup and prefix == 'probe':
        res = json.load(open(sup[0]))
        cells = [ms([100 * r['top1_acc'] for r in res if r['frac'] == f]) for f in (0.1, 0.5, 1.0)]
        f1 = ms([100 * r['macro_f1'] for r in res if r['frac'] == 0.1])
        print(f"| Supervised ViT (scratch) | – | – | {' | '.join(cells)} | {f1} |")
    per = {}
    for key, label in [('final_vanilla', 'Vanilla MAE (λ=0)'), ('final_spectral', 'Spectral-MAE (λ=10, brightness)')]:
        summ = [json.load(open(f)) for f in sorted(glob.glob(f'runs/{key}_s*/summary.json'))]
        acc, f1 = probe_by_seed(key, prefix)
        per[key] = acc
        print(f"| {label} | {ms([s['final_val_masked_mse'] * 1e3 for s in summ])} | "
              f"{ms([s['final_val_ndvi_mae'] for s in summ], 4)} | "
              + " | ".join(ms(list(acc[f].values())) for f in (0.1, 0.5, 1.0))
              + f" | {ms(list(f1[0.1].values()))} |")
    print("\nAccuracy rows: linear-probe mean ± std over 3 pre-training seeds (each averaged over 3 probe seeds);")
    print("supervised and random rows: mean ± std over 3 seeds.")
    print("\nPaired difference, Spectral − Vanilla, per pre-training seed:")
    for f in (0.1, 0.5, 1.0):
        common = sorted(set(per['final_spectral'][f]) & set(per['final_vanilla'][f]))
        d = [per['final_spectral'][f][s] - per['final_vanilla'][f][s] for s in common]
        print(f"  {int(f * 100):3d}% labels: {[round(float(x), 2) for x in d]}  mean {np.mean(d):+.2f}")
for f in sorted(glob.glob('runs/diag_final_*/per_class_gradients.md')):
    print('\n==', f); print(open(f).read())


### [CLS] token probe (spec)

| Model | Val MSE (×1e-3) | Val NDVI err | Acc 10% | Acc 50% | Acc 100% | Macro-F1 10% |
|---|---|---|---|---|---|---|
| Vanilla MAE (λ=0) | 1.51 ± 0.01 | 0.1173 ± 0.0001 | 66.34 ± 0.68 | 71.49 ± 0.45 | 72.79 ± 0.38 | 64.96 ± 0.72 |
| Spectral-MAE (λ=10, brightness) | 1.56 ± 0.01 | 0.1114 ± 0.0003 | 66.72 ± 0.59 | 70.96 ± 0.23 | 72.48 ± 0.15 | 65.37 ± 0.64 |

Accuracy rows: linear-probe mean ± std over 3 pre-training seeds (each averaged over 3 probe seeds);
supervised and random rows: mean ± std over 3 seeds.

Paired difference, Spectral − Vanilla, per pre-training seed:
   10% labels: [1.5, -0.96, 0.61]  mean +0.38
   50% labels: [-0.43, -1.24, 0.07]  mean -0.53
  100% labels: [-0.46, -0.74, 0.28]  mean -0.31

### Mean-pooled token probe

| Model | Val MSE (×1e-3) | Val NDVI err | Acc 10% | Acc 50% | Acc 100% | Macro-F1 10% |
|---|---|---|---|---|---|---|
| Vanilla MAE (λ=0) | 1.51 ± 0.01 | 0.1173 ± 0.0001 | 65.81 ± 0.88 | 70.64 ± 0.56 | 72.08 ± 0.60 | 

In [9]:
# 9. Pack everything for download (Output tab -> final_runs.zip)
!zip -qr /kaggle/working/final_runs.zip runs
print('Saved /kaggle/working/final_runs.zip')

Saved /kaggle/working/final_runs.zip
